In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path 

# Generate all 0-180 single and symmetric distractor conditions for threshold experiments 
All signals at 0 elevation.      

### Will use anechoic room


In [17]:
## Get target and distractor azimuths from human stimuli generation parameters 
human_stim_dir = Path("/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024")

azimuths = np.arange(0, 181, 10)
elevations = [0]


# fix SNR and room index to match 
snrs = np.arange(-18,7,3)
all_conditions = list(itertools.product(azimuths, snrs))
room_ix = 0

# fix SNR and room index to match 

anechoic_dir = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval")
room_manifest = pd.read_pickle(anechoic_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (anechoic_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (anechoic_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

conditions = []
# add single-distractor conditions
conditions += [{'target_loc': [0, 0],
                'distract_loc': [dist_azim, 0], # (azimuth, elevation)
                'snr': snr,
                'symmetric_distractor': False,  # Single distractor test 
                'test_room_meta': get_room_meta_dict(room_ix)} for (dist_azim, snr) in
                (all_conditions)]

# add symmetric distractor conditions
conditions += [{'target_loc': [0, 0],
                'distract_loc': [dist_azim, 0], # (azimuth, elevation)
                'snr': snr,
                'symmetric_distractor': True,  # Single distractor test 
                'test_room_meta': get_room_meta_dict(room_ix)} for (dist_azim, snr) in
                (all_conditions)]

# add texture distractor conditions
conditions += [{'target_loc': [0, 0],
                'distract_loc': [dist_azim, 0], # (azimuth, elevation)
                'snr': snr,
                'symmetric_distractor': False,  # Single distractor test 
                'with_textures': True,  
                'test_room_meta': get_room_meta_dict(room_ix)} for (dist_azim, snr) in
                (all_conditions)]

# add symmetric texture distractor conditions
conditions += [{'target_loc': [0, 0],
                'distract_loc': [dist_azim, 0], # (azimuth, elevation)
                'snr': snr,
                'symmetric_distractor': True,  # Single distractor test 
                'with_textures': True,  
                'test_room_meta': get_room_meta_dict(room_ix)} for (dist_azim, snr) in
                (all_conditions)]

cond_dict = {ix: cond_record for ix, cond_record in enumerate(conditions)}

# Assert keys contiguous
assert np.all(np.array(list(cond_dict.keys())) == np.arange(len(cond_dict)))
print(len(cond_dict))

684


In [ ]:
with open('binaural_test_manifests/front_back_threshold_single_and_symmetric_dist_speech_and_texture.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)